# RTB DSP Training Pipeline

This notebook implements the training pipeline for the Real-Time Bidding (RTB) Demand-Side Platform (DSP) optimization engine.

## Mdoel Architecture
We use **Logistic Regression (SGDClassifier)** for both CTR and CVR prediction. 
- **CTR Model**: Predicts $P(click=1 | features)$. Trained on all impressions (with negative downsampling).
- **CVR Model**: Predicts $P(conversion=1 | click=1, features)$. Trained only on clicked impressions (delayed feedback).

While the *algorithm* is the same, the **weights** are different because they learn different probabilities training on different data slices.

## Steps:
1.  **Setup**: Install dependencies.
2.  **Data Generation**: Create synthetic data.
3.  **Preprocessing**: Clean and prepare data using Polars.
4.  **Feature Engineering**: Apply Hashing Trick and Cross-Features.
5.  **Model Training**: Train separate SGDClassifiers for CTR and CVR.
6.  **Export**: Save weights to CSV for Java inference.

In [ ]:
# Install dependencies
!pip install polars scikit-learn mmh3 numpy pandas

In [ ]:
import polars as pl
import numpy as np
import mmh3
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import log_loss, roc_auc_score
import os
import time

## 2. Synthetic Data Generation

In [ ]:
# Configuration
NUM_SAMPLES = 100000
OUTPUT_DIR = "./"

def generate_synthetic_data(n_samples):
    np.random.seed(42)
    data = {
        "click": np.random.randint(0, 2, n_samples),
        "conversion": np.random.randint(0, 2, n_samples),
        "banner_pos": np.random.randint(0, 3, n_samples),
        "site_id": [f"site_{i}" for i in np.random.randint(0, 100, n_samples)],
        "app_id": [f"app_{i}" for i in np.random.randint(0, 50, n_samples)],
        "device_model": [f"model_{i}" for i in np.random.randint(0, 20, n_samples)],
        "device_type": np.random.randint(0, 5, n_samples),
        "C1": np.random.randint(1000, 1010, n_samples),
        "C14": np.random.randint(0, 100, n_samples),
        "C15": np.random.randint(300, 320, n_samples),
        "C16": np.random.randint(50, 60, n_samples),
    }
    # Make conversion dependent on click (cannot convert without click in this simple model)
    data["conversion"] = data["click"] * data["conversion"]
    
    return pl.DataFrame(data)

print("Generating synthetic data...")
df = generate_synthetic_data(NUM_SAMPLES)
print(f"Generated {df.height} rows.")
df.head()

## 3. Preprocessing & Feature Engineering

- **Hashing Trick**: Map categorical features to a fixed size vector ($2^{20}$).
- **Cross-Features**: Combine features to capture interactions.

In [ ]:
HASH_SIZE = 2**20  # ~1 million features

def get_hashed_index(feature_name, value):
    # Create a string representation: "feature_name=value"
    feature_str = f"{feature_name}={value}"
    # Hash using mmh3 and modulo to get index
    return mmh3.hash(feature_str, seed=42, signed=False) % HASH_SIZE

def transform_batch(batch_df):
    X_indices = []
    X_values = []
    y_click = []
    y_conversion = []
    
    # Categorical columns to use
    cat_cols = ["site_id", "app_id", "device_model", "device_type", "C1", "C14", "C15", "C16"]
    
    rows = batch_df.to_dicts()
    for row in rows:
        indices = []
        
        # 1. Single Features
        for col in cat_cols:
            idx = get_hashed_index(col, row[col])
            indices.append(idx)
            
        # 2. Cross Features (Example: site_id x app_id)
        # Note: In real scenarios, choose meaningful interactions
        cross_val = f"{row['site_id']}_x_{row['app_id']}"
        indices.append(get_hashed_index("site_app_cross", cross_val))
        
        X_indices.append(indices)
        # Value is always 1 for categorical features in this sparse representation
        X_values.append([1.0] * len(indices))
        
        y_click.append(row["click"])
        y_conversion.append(row["conversion"])
        
    return np.array(X_indices), np.array(X_values), np.array(y_click), np.array(y_conversion)

print("Transformation function defined.")

## 4. Model Training (SGDClassifier)

We use `SGDClassifier` with `loss='log_loss'` which implements Logistic Regression with Stochastic Gradient Descent.

### Negative Downsampling
To handle class imbalance and speed up training, we downsample the negative class (non-clicks). 
We keep 10% of negatives and all positives.

**Important**: The Java inference engine must recalibrate the prediction using:
$p_{calibrated} = \frac{p}{p + (1-p)/w}$ where $w = 0.1$.

In [ ]:
# Initialize models
# Two separate instances of SGDClassifier
ctr_model = SGDClassifier(loss='log_loss', penalty='l2', alpha=0.0001, fit_intercept=False, learning_rate='optimal', random_state=42)
cvr_model = SGDClassifier(loss='log_loss', penalty='l2', alpha=0.0001, fit_intercept=False, learning_rate='optimal', random_state=42)

from scipy.sparse import csr_matrix

def train_models(df):
    X_idx, X_val, y_clk, y_cnv = transform_batch(df)
    
    # Construct CSR Matrix for the whole batch first
    n_samples = len(y_clk)
    n_features_per_sample = X_idx.shape[1]
    
    row_indices = np.repeat(np.arange(n_samples), n_features_per_sample)
    col_indices = X_idx.flatten()
    data = X_val.flatten()
    
    X_sparse = csr_matrix((data, (row_indices, col_indices)), shape=(n_samples, HASH_SIZE))
    
    # --- CTR Training with Negative Downsampling ---
    # Keep all positives (click=1)
    # Keep 10% of negatives (click=0)
    DOWNSAMPLE_RATE = 0.1
    
    pos_mask = y_clk == 1
    # Randomly select 10% of negatives
    neg_mask = (y_clk == 0) & (np.random.rand(n_samples) < DOWNSAMPLE_RATE)
    
    train_mask = pos_mask | neg_mask
    
    X_train_ctr = X_sparse[train_mask]
    y_train_ctr = y_clk[train_mask]
    
    print(f"Training CTR model on {X_train_ctr.shape[0]} samples (downsampled from {n_samples})")
    if X_train_ctr.shape[0] > 0:
        ctr_model.partial_fit(X_train_ctr, y_train_ctr, classes=[0, 1])
    
    # --- CVR Training ---
    # Train ONLY on clicks (P(Conversion | Click))
    clicked_mask = y_clk == 1
    
    X_train_cvr = X_sparse[clicked_mask]
    y_train_cvr = y_cnv[clicked_mask]
    
    print(f"Training CVR model on {X_train_cvr.shape[0]} samples (clicked only)")
    if X_train_cvr.shape[0] > 0:
        cvr_model.partial_fit(X_train_cvr, y_train_cvr, classes=[0, 1])
    
    # Evaluate on the full batch (just for logging, optional)
    if np.sum(y_clk) > 0 and np.sum(y_clk) < len(y_clk):
        try:
            pred = ctr_model.predict_proba(X_sparse)[:, 1]
            print(f"Batch CTR Log Loss (on full batch): {log_loss(y_clk, pred):.4f}")
        except:
            pass

print("Training models...")
train_models(df)
print("Training complete.")

## 5. Export Weights

We need to export the learned weights to CSV files so the Java inference engine can load them.
The format will be:
`weight` (one per line, corresponding to index 0 to HASH_SIZE-1)

In [ ]:
def export_weights(model, filename):
    # Ensure we have weights (if model wasn't trained due to empty data, init zero weights)
    if not hasattr(model, "coef_"):
        model.partial_fit(csr_matrix((1, HASH_SIZE)), [0], classes=[0, 1])
        
    weights = model.coef_[0]
    
    # Create a DataFrame for fast export
    df_weights = pl.DataFrame({
        "weight": weights
    })
    
    # Export without header, just the values
    df_weights.write_csv(filename, include_header=False)
    print(f"Exported {filename} with shape {df_weights.shape}")

export_weights(ctr_model, "ctr_weights.csv")
export_weights(cvr_model, "cvr_weights.csv")